# Predict car prices using catboost

## Import library

In [ ]:
import pandas as pd
import numpy as np
import ast
from catboost import CatBoostRegressor

## Helper function

### parse_list_to_string
To clean up colomn that contains list into one string.

For example: `"['Grand', 'Caravan']"` into  `Grand Caravan`

def parse_list_string(s):
    try:
        val = ast.literal_eval(str(s))
        if isinstance(val, list) and len(val) > 0:
            return " ".join(val)
        return str(s)
    except:
        return str(s)

### extract_history

Simplify the history section to 3 parts. 
Accidents count, tittle status, and owners.

In [ ]:
def extract_history(history_str):
    try:
        h = ast.literal_eval(str(history_str))
        accidents = int(h[0]) if str(h[0]).isdigit() else 0
        owners = int(h[4]) if len(h) > 4 and str(h[4]).isdigit() else 1
        title = str(h[3]) if len(h) > 3 else "Unknown"
        return accidents, owners, title
    except:
        return 0, 1, "Unknown"

### preprocess_data

Main preprocessing function that combines all feature engineering steps.

def preprocess_data(df):
    # Keep the V2 Age calculation (2024)
    df['age'] = 2024 - df['manufacture_year']
    df['model_clean'] = df['model'].apply(parse_list_string)
    
    def get_feat_len(x):
        try: return len(ast.literal_eval(x))
        except: return 0
    df['feature_count'] = df['features'].apply(get_feat_len)
    
    history_features = df['vehicle_history'].apply(extract_history)
    df['accidents'], df['owners'], df['title_status'] = zip(*history_features)
    
    # Fill Numeric NaNs with -1 (This is the only change to numeric logic)
    for col in ['mpg_city', 'mpg_hwy', 'mileage']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1)
    
    # Fill Categorical NaNs
    cat_cols = ['brand', 'model_clean', 'transmission_type', 'drivetrain', 'fuel_type', 'state', 'title_status']
    for col in cat_cols:
        df[col] = df[col].astype(str).fillna('missing')
        
    return df

## Load Data

Load the training and test datasets from CSV files.

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/kcvanguard-machine-learning-assignment/train.csv').dropna(subset=['price'])
test_df = pd.read_csv('/kaggle/input/competitions/kcvanguard-machine-learning-assignment/test.csv')

## Preprocess Data

Apply preprocessing function to both training and test datasets.

In [ ]:
train_df = preprocess_data(train_df)
test_df = preprocess_data(test_df)

## Define Features

Specify numeric and categorical features for the model.

In [ ]:
numeric_features = ['age', 'mileage', 'mpg_city', 'mpg_hwy', 'feature_count', 'accidents', 'owners']
categorical_features = ['brand', 'model_clean', 'transmission_type', 'drivetrain', 'fuel_type', 'state', 'title_status']

## Prepare Training and Test Data

Create feature matrices and identify categorical feature indices.

In [ ]:
X_train = train_df[numeric_features + categorical_features]
y_train = train_df['price']
X_test = test_df[numeric_features + categorical_features]

cat_features_indices = [X_train.columns.get_loc(c) for c in categorical_features]

## Define Model

Configure CatBoost regressor with hyperparameters.

In [ ]:
model = CatBoostRegressor(
    iterations=2000,        
    learning_rate=0.05,     
    depth=8,                
    loss_function='RMSE',   
    random_seed=42,
    verbose=500
)

## Train Model and Make Predictions

Train the model on the full dataset and generate predictions.

In [ ]:
# Train on 100% of data
print("Training on full dataset...")
model.fit(X_train, y_train, cat_features=cat_features_indices)

# Predict & Clip
predictions = np.maximum(model.predict(X_test), 500)

## Create Submission File

Generate the final submission CSV file with predictions.

In [ ]:
submission = pd.DataFrame({'id': test_df['id'], 'price': np.round(predictions, 2)})
submission.to_csv('submission.csv', index=False)
print("Done! Submission created.")